# 03. 가계금융복지조사로 보는 성공 인식의 경제적 배경

이 노트북은 KGSS에서 확인한 성공 인식 변화를 경제적 배경과 연결하기 위한 보조 분석입니다.

KGSS는 사람들이 성공 요인을 어떻게 인식하는지 보여줍니다. 반면 가계금융복지조사는 한국 가구의 소득, 자산, 순자산 구조가 어떻게 변했는지 보여줍니다. 두 자료를 함께 보면, 사람들이 왜 `부유한 집안`, `부모 교육`, `좋은 사람을 아는 것` 같은 조건을 더 중요하게 보게 되었는지 맥락을 이해할 수 있습니다.

주의: 이 노트북은 개인 단위 원자료를 저장하지 않습니다. 통계청이 공개한 공식 집계 통계표에서 필요한 값만 읽고, 집계표와 그림만 저장합니다.


## 1. 라이브러리 불러오기

분석에는 `pandas`, `numpy`, `matplotlib`, `pathlib.Path`를 사용합니다. 엑셀 파일은 `pandas.read_excel()`로 읽습니다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager


## 2. 경로와 한글 폰트 설정

가계금융복지조사 부록 통계표 원본은 저장소 안으로 복사하지 않습니다. 현재 로컬 다운로드 폴더에 있는 파일을 읽도록 설정합니다.

다른 컴퓨터에서 실행할 경우 `finance_file` 경로만 본인의 파일 위치에 맞게 바꾸면 됩니다.


In [ ]:
project_root = Path('..').resolve()
outputs_tables = project_root / 'outputs' / 'tables'
outputs_figures = project_root / 'outputs' / 'figures'
outputs_tables.mkdir(parents=True, exist_ok=True)
outputs_figures.mkdir(parents=True, exist_ok=True)

finance_candidates = [
    project_root / 'data/raw/household_finance_2025/2025년+가계금융복지조사+부록+통계표.xlsx',
    Path.home() / 'Downloads/2025년+가계금융복지조사+부록+통계표.xlsx',
]
finance_file = next((p for p in finance_candidates if p.exists()), None)
if finance_file is None:
    raise FileNotFoundError('가계금융복지조사 부록 통계표 파일을 data/raw/household_finance_2025 또는 Downloads에 두세요.')

if not finance_file.exists():
    raise FileNotFoundError(
        f'가계금융복지조사 부록 통계표 파일을 찾을 수 없습니다: {finance_file}\n'
        '다른 환경에서 실행한다면 finance_file 경로를 본인 파일 위치로 수정하세요.'
    )

font_candidates = [
    'AppleGothic',
    'Malgun Gothic',
    'NanumGothic',
    'NanumBarunGothic',
    'Noto Sans CJK KR',
    'Noto Sans KR',
    'Arial Unicode MS',
]

installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = next((font for font in font_candidates if font in installed_fonts), 'DejaVu Sans')
plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False

selected_font


## 3. 엑셀 파일의 시트 확인

먼저 부록 통계표 안에 어떤 시트가 있는지 확인합니다. 이번 분석에서는 다음 시트를 사용합니다.

- `통계표14`: 연도별 자산과 부채 추이
- `통계표15`: 연도별 소득 추이
- `통계표12`: 순자산, 가구소득 분위별 평균과 점유율


In [ ]:
xls = pd.ExcelFile(finance_file)
xls.sheet_names


## 4. 연도별 자산·순자산 추이 읽기

`통계표14`에는 2012년부터 2025년까지의 평균 자산, 부채, 순자산이 들어 있습니다.

여기서는 평균 순자산을 현재 소득과 다른 시간축을 가진 `축적 자산 조건`을 보여주는 지표로 사용합니다.


In [ ]:
asset_raw = pd.read_excel(
    finance_file,
    sheet_name='통계표14',
    header=None,
    skiprows=6,
    nrows=14,
    usecols='A:H',
)

asset_trend = asset_raw.copy()
asset_trend.columns = [
    'year_label',
    'asset_total',
    'financial_asset',
    'real_asset',
    'debt_total',
    'financial_debt',
    'rental_deposit_debt',
    'net_asset',
]
asset_trend['YEAR'] = asset_trend['year_label'].astype(str).str.extract(r'(\d{4})').astype(int)
asset_trend = asset_trend.drop(columns='year_label')
asset_trend = asset_trend[['YEAR', 'asset_total', 'financial_asset', 'real_asset', 'debt_total', 'net_asset']]

asset_trend


## 5. 연도별 소득 추이 읽기

`통계표15`에는 2011년부터 2024년까지의 평균 가구소득과 근로소득이 들어 있습니다.

2016년은 자료원 변경으로 `2016년a`, `2016년b`가 함께 제시됩니다. 이후 연도와의 비교를 위해 이 노트북에서는 행정자료 보완 기준인 `2016년b`를 사용하고 `2016년a`는 제외합니다.


In [ ]:
income_raw = pd.read_excel(
    finance_file,
    sheet_name='통계표15',
    header=None,
    skiprows=6,
    nrows=15,
    usecols='A:G',
)

income_trend = income_raw.copy()
income_trend.columns = [
    'year_label',
    'household_income',
    'labor_income',
    'business_income',
    'property_income',
    'public_transfer_income',
    'private_transfer_income',
]
income_trend['YEAR'] = income_trend['year_label'].astype(str).str.extract(r'(\d{4})').astype(int)
income_trend['marker'] = income_trend['year_label'].astype(str).str.extract(r'2016년([ab])')
income_trend = income_trend[~((income_trend['YEAR'] == 2016) & (income_trend['marker'] == 'a'))].copy()
income_trend = income_trend.drop(columns=['year_label', 'marker'])
income_trend = income_trend[['YEAR', 'household_income', 'labor_income', 'business_income', 'property_income']]

income_trend


## 6. 근로소득과 순자산을 같은 기준으로 비교하기

근로소득은 현재 노동시장에서 벌어들이는 소득에 가깝고, 순자산은 주택·축적 기간·생애주기 효과가 섞인 누적 자산에 가깝습니다.

두 지표의 단위와 규모가 다르기 때문에 2012년을 100으로 놓고 지수화합니다. 이렇게 하면 어느 쪽이 더 빠르게 변했는지 비교할 수 있습니다.


In [ ]:
finance_trend = pd.merge(
    income_trend[['YEAR', 'household_income', 'labor_income']],
    asset_trend[['YEAR', 'net_asset']],
    on='YEAR',
    how='inner',
)

base_year = 2012
base = finance_trend.loc[finance_trend['YEAR'] == base_year].iloc[0]
finance_trend['household_income_index_2012'] = finance_trend['household_income'] / base['household_income'] * 100
finance_trend['labor_income_index_2012'] = finance_trend['labor_income'] / base['labor_income'] * 100
finance_trend['net_asset_index_2012'] = finance_trend['net_asset'] / base['net_asset'] * 100
finance_trend['net_asset_to_labor_income_ratio'] = finance_trend['net_asset'] / finance_trend['labor_income']

finance_trend.to_csv(outputs_tables / '03_income_net_asset_index.csv', index=False, encoding='utf-8-sig')
finance_trend


## 7. 근로소득 지수와 순자산 지수 시각화

이 그림은 `월급이 자산을 따라갈 수 있었는가`라는 질문을 위한 보조 그림입니다.

단순한 인과관계를 주장하기 위한 그림은 아닙니다. KGSS에서 배경 조건의 중요도 인식이 상승한 사회경제적 맥락을 보여주는 역할을 합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {
    'labor': '#2F7D95',
    'net_asset': '#C65A3A',
}

ax.plot(
    finance_trend['YEAR'],
    finance_trend['labor_income_index_2012'],
    marker='o',
    linewidth=2.8,
    color=colors['labor'],
    label='근로소득 지수',
)
ax.plot(
    finance_trend['YEAR'],
    finance_trend['net_asset_index_2012'],
    marker='o',
    linewidth=2.8,
    color=colors['net_asset'],
    label='순자산 지수',
)

for _, row in finance_trend[finance_trend['YEAR'].isin([2012, 2018, 2024])].iterrows():
    ax.text(row['YEAR'], row['labor_income_index_2012'] + 3, f"{row['labor_income_index_2012']:.0f}",
            ha='center', va='bottom', color=colors['labor'], fontsize=10, fontweight='bold')
    ax.text(row['YEAR'], row['net_asset_index_2012'] - 5, f"{row['net_asset_index_2012']:.0f}",
            ha='center', va='top', color=colors['net_asset'], fontsize=10, fontweight='bold')

ax.axhline(100, color='#999999', linewidth=1, linestyle='--')
ax.set_title('근로소득과 순자산은 어떻게 변했나?\n2012년=100으로 본 현재 소득과 축적 자산 조건', fontsize=15, fontweight='bold', pad=18)
ax.set_xlabel('연도')
ax.set_ylabel('지수 (2012년=100)')
ax.set_xticks(finance_trend['YEAR'])
ax.set_ylim(90, max(finance_trend['net_asset_index_2012'].max(), finance_trend['labor_income_index_2012'].max()) + 18)
ax.legend(frameon=False, loc='upper left')
ax.grid(axis='y', color='#E6E6E6')
ax.spines[['top', 'right']].set_visible(False)

source_note = '자료: 통계청, 2025년 가계금융복지조사 부록 통계표 14·15. 단위: 평균, 만원.'
fig.text(0.01, 0.01, source_note, ha='left', va='bottom', fontsize=9, color='#666666')
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(outputs_figures / '03_income_net_asset_index.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. 소득 격차와 순자산 격차 비교

`통계표12`는 2025년 순자산 5분위별 평균과 2024년 가구소득 5분위별 평균을 보여줍니다.

여기서는 상위 20% 평균을 하위 20% 평균으로 나눈 값을 비교합니다. 이 값이 클수록 상위와 하위의 차이가 크다는 뜻입니다.


In [ ]:
quintile_raw = pd.read_excel(
    finance_file,
    sheet_name='통계표 12',
    header=None,
    skiprows=5,
    nrows=6,
    usecols='A:J',
)

net_asset_bottom = float(quintile_raw.iloc[1, 3])
net_asset_top = float(quintile_raw.iloc[5, 3])
income_bottom = float(quintile_raw.iloc[1, 8])
income_top = float(quintile_raw.iloc[5, 8])

quintile_gap = pd.DataFrame([
    {
        'metric': '순자산',
        'year': 2025,
        'bottom_20_avg': net_asset_bottom,
        'top_20_avg': net_asset_top,
        'top_bottom_ratio': net_asset_top / net_asset_bottom,
        'unit': '만원',
        'source_table': '통계표12',
    },
    {
        'metric': '가구소득',
        'year': 2024,
        'bottom_20_avg': income_bottom,
        'top_20_avg': income_top,
        'top_bottom_ratio': income_top / income_bottom,
        'unit': '만원',
        'source_table': '통계표12',
    },
])

quintile_gap.to_csv(outputs_tables / '03_asset_income_quintile_gap.csv', index=False, encoding='utf-8-sig')
quintile_gap


## 9. 순자산 격차와 소득 격차 시각화

이 그림은 우리 프로젝트에서 가장 중요한 경제적 배경 그림입니다.

소득 격차도 존재하지만, 순자산 격차는 훨씬 크게 나타납니다. 따라서 순자산 격차는 성공을 현재 소득만으로 설명하기 어려운 경제적 배경으로 제시할 수 있습니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.8))
fig.patch.set_facecolor('white')

cards = [
    {
        'metric': '가구소득',
        'bottom': income_bottom,
        'top': income_top,
        'ratio': income_top / income_bottom,
        'color': '#5CA4A9',
        'subtitle': '1년에 벌어들인 소득',
    },
    {
        'metric': '순자산',
        'bottom': net_asset_bottom,
        'top': net_asset_top,
        'ratio': net_asset_top / net_asset_bottom,
        'color': '#C65A3A',
        'subtitle': '자산에서 부채를 뺀 축적 자산',
    },
]

def money_label(value):
    if value >= 10000:
        return f"{value/10000:.1f}억원"
    return f"{value:,.0f}만원"

for ax, card in zip(axes, cards):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    ax.text(0.5, 0.93, f"{card['metric']} 격차", ha='center', va='center', fontsize=17, fontweight='bold')
    ax.text(0.5, 0.85, card['subtitle'], ha='center', va='center', fontsize=12.5, color='#555555')

    # Bottom 20% box
    ax.text(0.18, 0.56, '하위 20% 평균', ha='center', va='center', fontsize=12, color='#555555')
    ax.text(0.18, 0.44, money_label(card['bottom']), ha='center', va='center', fontsize=21, fontweight='bold', color='#333333')

    # Arrow
    ax.annotate(
        '',
        xy=(0.68, 0.47),
        xytext=(0.32, 0.47),
        arrowprops=dict(arrowstyle='-|>', lw=2.5, color=card['color'], shrinkA=0, shrinkB=0),
    )

    # Top 20% box
    ax.text(0.82, 0.56, '상위 20% 평균', ha='center', va='center', fontsize=12, color='#555555')
    ax.text(0.82, 0.44, money_label(card['top']), ha='center', va='center', fontsize=21, fontweight='bold', color=card['color'])

    # Ratio badge
    ax.text(
        0.5,
        0.23,
        f"{card['ratio']:.1f}배 차이",
        ha='center',
        va='center',
        fontsize=22,
        fontweight='bold',
        color='white',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=card['color'], edgecolor='none'),
    )

    ax.text(
        0.5,
        0.10,
        f"상위 20% 평균 ÷ 하위 20% 평균",
        ha='center',
        va='center',
        fontsize=10.5,
        color='#777777',
    )

fig.suptitle('상위·하위 격차는 소득보다 순자산에서 훨씬 크게 나타난다', fontsize=18, fontweight='bold', y=0.99)
fig.text(0.01, 0.01, '자료: 통계청, 2025년 가계금융복지조사 부록 통계표 12. 단위: 평균, 만원. 순자산은 2025년, 가구소득은 2024년 기준.',
         ha='left', va='bottom', fontsize=9, color='#666666')
fig.tight_layout(rect=[0, 0.05, 1, 0.94])
fig.savefig(outputs_figures / '03_asset_income_quintile_gap.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. 배수 비교 그래프도 함께 저장하기

카드형 그림은 발표 슬라이드에서 메시지를 빠르게 전달하기 좋습니다. 다만 분석 결과를 그래프로 보여주기 위해, 같은 값을 막대그래프로도 저장합니다.

이 그래프는 `상위 20% 평균 ÷ 하위 20% 평균`을 직접 비교합니다.


In [ ]:
ratio_chart = quintile_gap.copy()
ratio_chart['metric'] = pd.Categorical(ratio_chart['metric'], categories=['가구소득', '순자산'], ordered=True)
ratio_chart = ratio_chart.sort_values('metric')

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#5CA4A9' if metric == '가구소득' else '#C65A3A' for metric in ratio_chart['metric']]
bars = ax.bar(ratio_chart['metric'].astype(str), ratio_chart['top_bottom_ratio'], color=colors, width=0.5)

for bar, (_, row) in zip(bars, ratio_chart.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 4,
        f"{row['top_bottom_ratio']:.1f}배",
        ha='center',
        va='bottom',
        fontsize=16,
        fontweight='bold',
        color='#333333',
    )

asset_ratio = ratio_chart.loc[ratio_chart['metric'].astype(str) == '순자산', 'top_bottom_ratio'].iloc[0]
income_ratio = ratio_chart.loc[ratio_chart['metric'].astype(str) == '가구소득', 'top_bottom_ratio'].iloc[0]
ratio_gap = asset_ratio / income_ratio

ax.annotate(
    f"순자산 격차는\n가구소득 격차의 약 {ratio_gap:.1f}배",
    xy=(1, asset_ratio),
    xytext=(0.45, asset_ratio * 0.72),
    arrowprops=dict(arrowstyle='->', color='#555555', lw=1.8),
    fontsize=12.5,
    ha='center',
    va='center',
    bbox=dict(boxstyle='round,pad=0.45', facecolor='#F2F4F4', edgecolor='none'),
)

ax.set_title('상위·하위 격차는 소득보다 순자산에서 훨씬 크다\n상위 20% 평균 ÷ 하위 20% 평균', fontsize=15, fontweight='bold', pad=18)
ax.set_ylabel('배수')
ax.set_ylim(0, asset_ratio * 1.22)
ax.grid(axis='y', color='#E6E6E6')
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', labelsize=13)
ax.tick_params(axis='y', labelsize=10)

fig.text(0.01, 0.01, '자료: 통계청, 2025년 가계금융복지조사 부록 통계표 12. 순자산은 2025년, 가구소득은 2024년 기준.',
         ha='left', va='bottom', fontsize=9, color='#666666')
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig(outputs_figures / '03_asset_income_quintile_gap_ratio_chart.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. 실제 금액 기준 상위·하위 비교 그래프

카드형 그림에 들어간 실제 금액을 그래프 형태로도 저장합니다. 이 그래프는 `하위 20% 평균`과 `상위 20% 평균`의 금액 차이를 직접 보여줍니다.

금액 차이가 매우 크기 때문에 y축은 로그축을 사용합니다. 단, 보조 격자선과 연결선은 제거해 해석을 방해하지 않도록 했습니다.


In [ ]:
amount_df = pd.DataFrame([
    {'metric': '가구소득', 'group': '하위 20% 평균', 'amount': income_bottom, 'color': '#D9DEE2'},
    {'metric': '가구소득', 'group': '상위 20% 평균', 'amount': income_top, 'color': '#5CA4A9'},
    {'metric': '순자산', 'group': '하위 20% 평균', 'amount': net_asset_bottom, 'color': '#D9DEE2'},
    {'metric': '순자산', 'group': '상위 20% 평균', 'amount': net_asset_top, 'color': '#C65A3A'},
])

fig, axes = plt.subplots(1, 2, figsize=(12, 5.8), sharey=True)

def money_label(value):
    if value >= 10000:
        return f"{value/10000:.1f}억원"
    return f"{value:,.0f}만원"

def money_tick(value, pos):
    if value >= 10000:
        return f"{value/10000:.0f}억원"
    return f"{value/1000:.0f}천만원"

for ax, metric in zip(axes, ['가구소득', '순자산']):
    data = amount_df[amount_df['metric'] == metric].copy()
    bars = ax.bar(data['group'], data['amount'], color=data['color'], width=0.52)
    ax.set_yscale('log')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(money_tick))
    ax.grid(axis='y', which='major', color='#E6E6E6')
    ax.grid(axis='y', which='minor', visible=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=10)

    ratio = data.loc[data['group'] == '상위 20% 평균', 'amount'].iloc[0] / data.loc[data['group'] == '하위 20% 평균', 'amount'].iloc[0]
    subtitle = '1년에 벌어들인 소득' if metric == '가구소득' else '자산에서 부채를 뺀 축적 자산'
    ax.set_title(f"{metric}: {ratio:.1f}배 차이\n{subtitle}", fontsize=14, fontweight='bold', pad=14)

    for bar, value in zip(bars, data['amount']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value * 1.12,
            money_label(value),
            ha='center',
            va='bottom',
            fontsize=12,
            fontweight='bold',
            color='#333333',
        )

axes[0].set_ylabel('평균 금액 (로그축)')
fig.suptitle('하위 20%와 상위 20%의 금액 차이', fontsize=17, fontweight='bold', y=1.03)
fig.text(0.01, 0.01, '자료: 통계청, 2025년 가계금융복지조사 부록 통계표 12. 단위: 평균, 만원. 순자산은 2025년, 가구소득은 2024년 기준.',
         ha='left', va='bottom', fontsize=9, color='#666666')
fig.tight_layout(rect=[0, 0.05, 1, 0.98])
fig.savefig(outputs_figures / '03_asset_income_quintile_gap_amount_chart.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. KGSS 성공 인식 결과와 함께 보기

마지막으로 02번 노트북에서 만든 KGSS 성공 인식 집계표를 함께 불러옵니다. 이 표는 개인 단위 자료가 아니라 이미 저장된 집계표입니다.

여기서는 2025년 성공 요인 중요도와 가계금융복지조사의 자산·소득 격차를 한 문장으로 연결해 봅니다.


In [ ]:
success_by_year_path = outputs_tables / '02_success_importance_by_year.csv'
if success_by_year_path.exists():
    success_2025 = pd.read_csv(success_by_year_path)
    success_2025 = success_2025[success_2025['YEAR'] == 2025].copy()
    success_2025 = success_2025[['label', 'weighted_important_pct', 'valid_n']]
    display(success_2025)
else:
    print('02_success_importance_by_year.csv 파일이 아직 없습니다. 02번 노트북을 먼저 실행하세요.')


## 11. KGSS 인식과 자산 격차를 한 장에서 연결하기

아래 그림은 발표용 후보입니다. 왼쪽은 KGSS 2025년 성공 요인 중요도이고, 오른쪽은 가계금융복지조사의 소득·순자산 분위 격차입니다.

이 그림의 메시지는 단순합니다. 한국인은 여전히 노력을 중요하게 보지만, 성공을 현재 소득만으로 설명하기 어렵게 만드는 축적 자산 격차도 크게 존재한다는 점입니다.


In [ ]:
if success_by_year_path.exists():
    success_2025_plot = pd.read_csv(success_by_year_path)
    success_2025_plot = success_2025_plot[success_2025_plot['YEAR'] == 2025].copy()
    label_order = ['열심히 일', '좋은 사람을 아는 것', '부유한 집안', '부모 교육']
    success_2025_plot['label'] = pd.Categorical(success_2025_plot['label'], categories=label_order, ordered=True)
    success_2025_plot = success_2025_plot.sort_values('label', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1.2, 1]})

    ax = axes[0]
    colors_left = success_2025_plot['label'].map({
        '열심히 일': '#2F7D95',
        '좋은 사람을 아는 것': '#2F7D95',
        '부유한 집안': '#C65A3A',
        '부모 교육': '#C65A3A',
    })
    ax.barh(success_2025_plot['label'].astype(str), success_2025_plot['weighted_important_pct'], color=colors_left, height=0.52)
    for _, row in success_2025_plot.iterrows():
        ax.text(row['weighted_important_pct'] + 0.45, str(row['label']), f"{row['weighted_important_pct']:.1f}%",
                va='center', ha='left', fontsize=11, fontweight='bold')
    ax.set_xlim(80, 102)
    ax.set_xlabel('중요하다고 본 비율 (%)')
    ax.set_title('2025년 성공 요인 인식\n노력은 여전히 높지만 배경 조건도 높다', fontsize=13, fontweight='bold', pad=14)
    ax.grid(axis='x', color='#E6E6E6')
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='y', length=0)

    ax = axes[1]
    plot_df = quintile_gap.sort_values('top_bottom_ratio')
    bar_colors = ['#5CA4A9' if metric == '가구소득' else '#C65A3A' for metric in plot_df['metric']]
    bars = ax.barh(plot_df['metric'], plot_df['top_bottom_ratio'], color=bar_colors, height=0.52)
    for bar, (_, row) in zip(bars, plot_df.iterrows()):
        ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2, f"{row['top_bottom_ratio']:.1f}배",
                va='center', ha='left', fontsize=12, fontweight='bold')
    ax.set_xlim(0, max(plot_df['top_bottom_ratio']) * 1.22)
    ax.set_xlabel('상위 20% 평균 / 하위 20% 평균')
    ax.set_title('소득보다 더 크게 벌어진 순자산 격차\n순자산은 현재 소득과 다른 시간축을 가진다', fontsize=13, fontweight='bold', pad=14)
    ax.grid(axis='x', color='#E6E6E6')
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='y', length=0)

    fig.suptitle('노력은 중요하지만, 축적 자산 조건도 함께 보인다', fontsize=17, fontweight='bold', y=1.03)
    fig.text(0.01, 0.01, '자료: KGSS 2003-2025 누적자료; 통계청, 2025년 가계금융복지조사 부록 통계표 12.',
             ha='left', va='bottom', fontsize=9, color='#666666')
    fig.tight_layout(rect=[0, 0.04, 1, 0.98])
    fig.savefig(outputs_figures / '03_success_perception_asset_context.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('02_success_importance_by_year.csv 파일이 없어 연결 그림을 만들 수 없습니다.')


## 12. 연령대별 소득과 순자산도 함께 확인하기

전체 가구를 기준으로 순자산 상위·하위 격차를 비교하면 생애주기 효과가 섞일 수 있습니다. 예를 들어 50대와 60대는 20대와 30대보다 자산을 축적할 시간이 길었기 때문에 순자산이 높게 나타날 수 있습니다.

따라서 전체 분위 비교는 `전체 구조`를 보여주는 자료로 사용하고, 연령대별 소득과 순자산을 함께 확인해 해석을 보완합니다.


In [ ]:
# 통계표1~10은 여러 표가 한 시트에 가로로 붙어 있습니다.
# 아래 열 위치는 2025년 부록 통계표 구조를 기준으로 필요한 최신값만 읽습니다.
# 자산/부채/순자산은 2025년 기준, 가구소득은 조사에서 제시된 최신 소득 기준입니다.

age_row_map = {
    '39세 이하': 16,
    '40대': 19,
    '50대': 20,
    '60세 이상': 21,
}

age_records = []
for age_group, excel_row in age_row_map.items():
    row = excel_row - 1  # pandas는 0-index
    sheet = pd.read_excel(finance_file, sheet_name='통계표1~10', header=None)
    age_records.append({
        'age_group': age_group,
        'asset_total': sheet.iloc[row, 9],      # 자산 최신값
        'debt_total': sheet.iloc[row, 69],      # 부채 최신값
        'net_asset': sheet.iloc[row, 111],      # 순자산 최신값
        'household_income': sheet.iloc[row, 117], # 가구소득 최신값
    })

age_finance = pd.DataFrame(age_records)
age_finance['net_asset_to_income_ratio'] = age_finance['net_asset'] / age_finance['household_income']
age_finance.to_csv(outputs_tables / '03_age_income_net_asset_context.csv', index=False, encoding='utf-8-sig')
age_finance


## 13. 연령대별 소득과 순자산 시각화

이 그림은 순자산 격차를 해석할 때 나이를 고려해야 한다는 점을 보여줍니다. 순자산은 한 해 소득이 아니라 오랜 기간 누적된 자산이기 때문에, 연령대가 높을수록 커질 수 있습니다.

따라서 발표에서는 전체 순자산 격차를 말할 때 `연령과 생애주기 효과가 섞여 있다`는 한계를 함께 적어야 합니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.8))

x = np.arange(len(age_finance))
bar_width = 0.58

# 왼쪽: 가구소득과 순자산 금액 비교
ax = axes[0]
ax.bar(x - bar_width/4, age_finance['household_income'], width=bar_width/2, color='#5CA4A9', label='가구소득')
ax.bar(x + bar_width/4, age_finance['net_asset'], width=bar_width/2, color='#C65A3A', label='순자산')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, p: f'{v/10000:.0f}억원' if v >= 10000 else f'{v/1000:.0f}천만원'))
ax.set_xticks(x)
ax.set_xticklabels(age_finance['age_group'])
ax.set_ylabel('평균 금액 (로그축)')
ax.set_title('연령대별 평균 가구소득과 순자산\n순자산은 생애주기 영향을 함께 받는다', fontsize=13.5, fontweight='bold', pad=14)
ax.legend(frameon=False)
ax.grid(axis='y', which='major', color='#E6E6E6')
ax.grid(axis='y', which='minor', visible=False)
ax.spines[['top', 'right']].set_visible(False)

for i, row in age_finance.iterrows():
    ax.text(i + bar_width/4, row['net_asset'] * 1.12, f"{row['net_asset']/10000:.1f}억", ha='center', va='bottom', fontsize=9.5, color='#C65A3A', fontweight='bold')

# 오른쪽: 순자산 / 가구소득 비율
ax = axes[1]
ax.plot(age_finance['age_group'], age_finance['net_asset_to_income_ratio'], marker='o', color='#6B4C9A', linewidth=2.8)
for _, row in age_finance.iterrows():
    ax.text(row['age_group'], row['net_asset_to_income_ratio'] + 0.25, f"{row['net_asset_to_income_ratio']:.1f}배",
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#6B4C9A')
ax.set_title('순자산은 연령대가 높을수록\n소득 대비 더 크게 누적되어 있다', fontsize=13.5, fontweight='bold', pad=14)
ax.set_ylabel('순자산 / 가구소득')
ax.set_ylim(0, age_finance['net_asset_to_income_ratio'].max() + 1.4)
ax.grid(axis='y', color='#E6E6E6')
ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('순자산 격차에는 연령과 생애주기 효과가 함께 섞여 있다', fontsize=17, fontweight='bold', y=1.03)
fig.text(0.01, 0.01, '자료: 통계청, 2025년 가계금융복지조사 부록 통계표 1~10. 단위: 평균, 만원.',
         ha='left', va='bottom', fontsize=9, color='#666666')
fig.tight_layout(rect=[0, 0.05, 1, 0.98])
fig.savefig(outputs_figures / '03_age_income_net_asset_context.png', dpi=200, bbox_inches='tight')
plt.show()


## 12. 해석 정리

이 노트북의 결과는 다음과 같이 해석할 수 있습니다.

1. KGSS에서 한국인은 여전히 `열심히 일`을 성공의 핵심 요인으로 본다.
2. 하지만 `부유한 집안`, `부모 교육`, `좋은 사람을 아는 것` 같은 배경·관계 조건의 중요도 인식도 상승했다.
3. 가계금융복지조사에서는 소득 격차보다 순자산 격차가 훨씬 크게 나타난다.
4. 따라서 한국인이 잃은 것은 노력의 가치라기보다, 성공을 노력만으로 설명할 수 있다는 단순한 공식에 대한 신뢰일 수 있다.

주의할 점은 이 분석이 인과관계를 증명하지 않는다는 것입니다. 가계금융복지조사는 KGSS 응답자와 직접 연결된 자료가 아니므로, 경제적 배경을 설명하는 맥락 자료로 사용해야 합니다.
